# Answer Key to the Data Wrangling with DataFrames Coding Quiz

Helpful resources:
https://spark.apache.org/docs/2.4.0/api/python/pyspark.sql.html

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import isnan, count, when, col, desc, udf, col, sort_array, asc, avg
from pyspark.sql.functions import sum as Fsum
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType

In [4]:
# 1) import any other libraries you might need
# 2) instantiate a Spark session 
# 3) read in the data set located at the path "data/sparkify_log_small.json"
# 4) write code to answer the quiz questions 

spark = SparkSession \
    .builder \
    .appName("Data Frames practice") \
    .getOrCreate()

df = spark.read.json("./data/music_log_small.json")

# Question 1

Which page did user id "" (empty string) NOT visit?

In [5]:
df.printSchema()

root
 |-- artist: string (nullable = true)
 |-- auth: string (nullable = true)
 |-- firstName: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- itemInSession: long (nullable = true)
 |-- lastName: string (nullable = true)
 |-- length: double (nullable = true)
 |-- level: string (nullable = true)
 |-- location: string (nullable = true)
 |-- method: string (nullable = true)
 |-- page: string (nullable = true)
 |-- registration: long (nullable = true)
 |-- sessionId: long (nullable = true)
 |-- song: string (nullable = true)
 |-- status: long (nullable = true)
 |-- ts: long (nullable = true)
 |-- userAgent: string (nullable = true)
 |-- userId: string (nullable = true)



In [7]:
df.select("page").distinct().subtract(df.filter(df.userId == "").select("page").distinct()).show()

+----------------+
|            page|
+----------------+
|Submit Downgrade|
|       Downgrade|
|          Logout|
|   Save Settings|
|        Settings|
|        NextSong|
|         Upgrade|
|           Error|
|  Submit Upgrade|
+----------------+



# Question 2 - Reflect

What type of user does the empty string user id most likely refer to?

In [17]:
#What websites did the user visit?
df.filter(df.userId == "").select("page").distinct().show()

+-----+
| page|
+-----+
| Home|
|About|
|Login|
| Help|
+-----+



Considering the empty string user only visited sites like "Home" and "Login", all sites that can be accessed without logging in, and did not visit sites that could presumably only be visited when logged in, I assume the empty string user most likely refers to all users, that are browsing the site without being logged in.

# Question 3

How many female users do we have in the data set?

In [16]:
# checking how the dataset handles this info
df.select("gender").distinct().show()

+------+
|gender|
+------+
|     F|
|     M|
|  NULL|
+------+



In [15]:
df.filter(df.gender == "F").count()

3820

# Question 4

How many songs were played from the most played artist?

In [20]:
df.filter(df.page == "NextSong") \
    .groupBy("artist")  \
    .agg(count("song").alias("song_count")) \
    .orderBy(desc("song_count")) \
    .first()

Row(artist='Coldplay', song_count=83)

# Question 5 (challenge)

How many songs do users listen to on average between visiting our home page? Please round your answer to the closest integer.

In [21]:
df.filter(df.page == "NextSong").groupBy("sessionId", "userId").agg(count("song").alias("num_songs")).select(avg("num_songs")).first()[0]

6.830605564648118